In [1]:
import pandas as pd
import math

In [2]:
# ============================================================
# 1. CREATE DATASET
# ============================================================

data = [
    ["S1",  "High",   "High",   "Yes", "High", "Yes", "Yes"],
    ["S2",  "High",   "High",   "Yes", "Low",  "Yes", "Yes"],
    ["S3",  "Medium", "High",   "Yes", "High", "Yes", "Yes"],
    ["S4",  "Low",    "Low",    "No",  "High", "No",  "No"],
    ["S5",  "Medium", "Low",    "No",  "High", "Yes", "No"],
    ["S6",  "High",   "Low",    "No",  "Low",  "Yes", "Yes"],
    ["S7",  "Low",    "High",   "Yes", "Low",  "No",  "No"],
    ["S8",  "Medium", "High",   "No",  "High", "Yes", "Yes"],
    ["S9",  "Low",    "Low",    "Yes", "High", "No",  "No"],
    ["S10", "High",   "Medium", "Yes", "High", "Yes", "Yes"],
    ["S11", "Medium", "Medium", "No",  "Low",  "No",  "No"],
    ["S12", "High",   "Medium", "No",  "High", "Yes", "Yes"]
]

columns = [
    "Student",
    "Budget",
    "ScreenTime",
    "Gaming",
    "CameraPriority",
    "5G",
    "BuyPhone"
]

df = pd.DataFrame(data, columns=columns)

print("=" * 70)
print("SMARTPHONE PURCHASE DATASET")
print("=" * 70)

print(df.to_string(index=False))



SMARTPHONE PURCHASE DATASET
Student Budget ScreenTime Gaming CameraPriority  5G BuyPhone
     S1   High       High    Yes           High Yes      Yes
     S2   High       High    Yes            Low Yes      Yes
     S3 Medium       High    Yes           High Yes      Yes
     S4    Low        Low     No           High  No       No
     S5 Medium        Low     No           High Yes       No
     S6   High        Low     No            Low Yes      Yes
     S7    Low       High    Yes            Low  No       No
     S8 Medium       High     No           High Yes      Yes
     S9    Low        Low    Yes           High  No       No
    S10   High     Medium    Yes           High Yes      Yes
    S11 Medium     Medium     No            Low  No       No
    S12   High     Medium     No           High Yes      Yes


In [3]:
# ============================================================
# 2. SEPARATE POSITIVE AND NEGATIVE EXAMPLES
# ============================================================

positive_examples = df[df["BuyPhone"] == "Yes"].copy()
negative_examples = df[df["BuyPhone"] == "No"].copy()

print("\n" + "=" * 70)
print("POSITIVE EXAMPLES")
print("=" * 70)

print(positive_examples[["Student"]].to_string(index=False))

print("\n" + "=" * 70)
print("NEGATIVE EXAMPLES")
print("=" * 70)

print(negative_examples[["Student"]].to_string(index=False))



POSITIVE EXAMPLES
Student
     S1
     S2
     S3
     S6
     S8
    S10
    S12

NEGATIVE EXAMPLES
Student
     S4
     S5
     S7
     S9
    S11


In [4]:
# ============================================================
# 3. FOIL GAIN FUNCTION
# ============================================================

def foil_gain(p0, n0, p1, n1):
    """
    Calculate FOIL Information Gain.

    p0 = positive examples before adding literal
    n0 = negative examples before adding literal
    p1 = positive examples after adding literal
    n1 = negative examples after adding literal
    """

    # Cannot learn from a condition that covers no positives
    if p1 == 0:
        return float("-inf")

    # If no negative examples are covered,
    # the rule is perfectly specific.
    if n1 == 0:
        new_probability = 1.0
    else:
        new_probability = p1 / (p1 + n1)

    old_probability = p0 / (p0 + n0)

    # Number of positive examples covered
    t = p1

    gain = t * (
        math.log2(new_probability)
        - math.log2(old_probability)
    )

    return gain

In [5]:
# ============================================================
# 4. GENERATE CANDIDATE LITERALS
# ============================================================

features = [
    "Budget",
    "ScreenTime",
    "Gaming",
    "CameraPriority",
    "5G"
]


def get_candidates(current_positive, current_negative):
    """
    Generate all possible feature=value conditions.
    """

    candidates = []

    p0 = len(current_positive)
    n0 = len(current_negative)

    for feature in features:

        values = df[feature].unique()

        for value in values:

            positive_match = current_positive[
                current_positive[feature] == value
            ]

            negative_match = current_negative[
                current_negative[feature] == value
            ]

            p1 = len(positive_match)
            n1 = len(negative_match)

            gain = foil_gain(
                p0,
                n0,
                p1,
                n1
            )

            candidates.append({
                "feature": feature,
                "value": value,
                "p1": p1,
                "n1": n1,
                "gain": gain
            })

    return candidates


In [6]:
# ============================================================
# 5. LEARN ONE FOIL RULE
# ============================================================

def learn_one_rule(positive, negative):

    rule = []

    current_positive = positive.copy()
    current_negative = negative.copy()

    print("\n" + "-" * 70)
    print("LEARNING A NEW RULE")
    print("-" * 70)

    while len(current_negative) > 0:

        candidates = get_candidates(
            current_positive,
            current_negative
        )

        # Remove conditions already used
        candidates = [
            c for c in candidates
            if (c["feature"], c["value"]) not in rule
        ]

        if not candidates:
            break

        # Select candidate with maximum FOIL gain
        best = max(
            candidates,
            key=lambda x: x["gain"]
        )

        # If no useful condition exists
        if best["gain"] == float("-inf"):
            break

        feature = best["feature"]
        value = best["value"]

        rule.append((feature, value))

        print(
            f"\nSelected condition: "
            f"{feature} = {value}"
        )

        print(
            f"Positive covered: {best['p1']}"
        )

        print(
            f"Negative covered: {best['n1']}"
        )

        print(
            f"FOIL Gain: {best['gain']:.4f}"
        )

        # Apply selected condition
        current_positive = current_positive[
            current_positive[feature] == value
        ]

        current_negative = current_negative[
            current_negative[feature] == value
        ]

    return rule


In [7]:
# ============================================================
# 6. SEQUENTIAL COVERING / FOIL
# ============================================================

def foil_learning():

    remaining_positive = positive_examples.copy()

    remaining_negative = negative_examples.copy()

    learned_rules = []

    print("\n\n")
    print("=" * 70)
    print("STARTING FOIL ALGORITHM")
    print("=" * 70)

    while len(remaining_positive) > 0:

        print(
            f"\nRemaining positive examples: "
            f"{len(remaining_positive)}"
        )

        # Learn one rule
        rule = learn_one_rule(
            remaining_positive,
            remaining_negative
        )

        if not rule:
            break

        learned_rules.append(rule)

        # ----------------------------------------------------
        # Find positive examples covered by the new rule
        # ----------------------------------------------------

        covered_positive = remaining_positive.copy()

        for feature, value in rule:

            covered_positive = covered_positive[
                covered_positive[feature] == value
            ]

        # ----------------------------------------------------
        # Remove covered positive examples
        # ----------------------------------------------------

        remaining_positive = remaining_positive.drop(
            covered_positive.index
        )

        print("\nRule learned:")
        print_rule(rule)

        print(
            "\nPositive examples covered:"
        )

        print(
            covered_positive["Student"]
            .tolist()
        )

    return learned_rules


In [8]:

# ============================================================
# 7. PRINT RULE
# ============================================================

def print_rule(rule):

    conditions = []

    for feature, value in rule:

        conditions.append(
            f"{feature}(X, {value})"
        )

    body = " AND ".join(conditions)

    print(
        f"BuyPhone(X) :- {body}"
    )


In [9]:

# ============================================================
# 8. RUN FOIL
# ============================================================

rules = foil_learning()






STARTING FOIL ALGORITHM

Remaining positive examples: 7

----------------------------------------------------------------------
LEARNING A NEW RULE
----------------------------------------------------------------------

Selected condition: 5G = Yes
Positive covered: 7
Negative covered: 1
FOIL Gain: 4.0947

Selected condition: Budget = High
Positive covered: 5
Negative covered: 0
FOIL Gain: 0.9632

Rule learned:
BuyPhone(X) :- 5G(X, Yes) AND Budget(X, High)

Positive examples covered:
['S1', 'S2', 'S6', 'S10', 'S12']

Remaining positive examples: 2

----------------------------------------------------------------------
LEARNING A NEW RULE
----------------------------------------------------------------------

Selected condition: ScreenTime = High
Positive covered: 2
Negative covered: 1
FOIL Gain: 2.4448

Selected condition: Budget = Medium
Positive covered: 2
Negative covered: 0
FOIL Gain: 1.1699

Rule learned:
BuyPhone(X) :- ScreenTime(X, High) AND Budget(X, Medium)

Positive exampl

In [10]:

# ============================================================
# 9. DISPLAY FINAL RULES
# ============================================================

print("\n\n")
print("=" * 70)
print("FINAL FOIL RULES")
print("=" * 70)

for i, rule in enumerate(rules, start=1):

    print(f"\nRule {i}:")
    print_rule(rule)





FINAL FOIL RULES

Rule 1:
BuyPhone(X) :- 5G(X, Yes) AND Budget(X, High)

Rule 2:
BuyPhone(X) :- ScreenTime(X, High) AND Budget(X, Medium)


In [11]:


# ============================================================
# 10. RULE-BASED PREDICTION
# ============================================================

def predict(student):

    """
    Predict BuyPhone using the learned FOIL rules.
    """

    for rule in rules:

        matches = True

        for feature, value in rule:

            if student[feature] != value:
                matches = False
                break

        if matches:
            return "Yes"

    return "No"


In [12]:


# ============================================================
# 11. TEST THE MODEL ON DATASET
# ============================================================

df["Predicted"] = df.apply(
    predict,
    axis=1
)

print("\n\n")
print("=" * 70)
print("PREDICTIONS")
print("=" * 70)

print(
    df[
        [
            "Student",
            "BuyPhone",
            "Predicted"
        ]
    ].to_string(index=False)
)







PREDICTIONS
Student BuyPhone Predicted
     S1      Yes       Yes
     S2      Yes       Yes
     S3      Yes       Yes
     S4       No        No
     S5       No        No
     S6      Yes       Yes
     S7       No        No
     S8      Yes       Yes
     S9       No        No
    S10      Yes       Yes
    S11       No        No
    S12      Yes       Yes


In [13]:
# ============================================================
# 12. CALCULATE ACCURACY
# ============================================================

correct = (
    df["BuyPhone"] ==
    df["Predicted"]
).sum()

total = len(df)

accuracy = (correct / total) * 100

print("\n" + "=" * 70)
print("MODEL PERFORMANCE")
print("=" * 70)

print(f"Correct predictions : {correct}")
print(f"Total examples      : {total}")
print(f"Accuracy            : {accuracy:.2f}%")


MODEL PERFORMANCE
Correct predictions : 12
Total examples      : 12
Accuracy            : 100.00%
